<a href="https://colab.research.google.com/github/sameerahmed-cloud/text2sql-small-llm/blob/main/notebooks/01_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
REPO_DIR = "/content/text2sql-small-llm"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sameerahmed-cloud/text2sql-small-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}

Cloning into '/content/text2sql-small-llm'...
remote: Enumerating objects: 105, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 105 (delta 49), reused 56 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (105/105), 150.23 KiB | 2.31 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/text2sql-small-llm


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import json
from src.get_data import unpack_all
from src.data_prep import DRIVE_PROCESSED

paths = unpack_all()   # BIRD dev + Spider: enough for the exams

def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]

tests = {n: read_jsonl(DRIVE_PROCESSED / f"{n}.jsonl") for n in ["test_bird", "test_spider"]}

RESULTS = "/content/drive/MyDrive/text2sql/results"
PREDS = f"{RESULTS}/preds"
FEWSHOT = DRIVE_PROCESSED / "fewshot_examples.json"
MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

Mounted at /content/drive
Unpacking bird_dev (big datasets take several minutes)...
Unpacking spider (big datasets take several minutes)...
✓ Paths saved to /content/data/raw/paths.json


In [ ]:
from src.evaluate import evaluate

RESULTS = "/content/drive/MyDrive/text2sql/results"   # on Drive, so the files survive the session

for name, recs in tests.items():
    gold_as_answers = {r["id"]: r["gold_sql"] for r in recs}   # pretend the model is perfect
    evaluate(recs, gold_as_answers, paths, run_id=f"sanity-gold-{name}", out_dir=RESULTS)

  graded 250/1534
  graded 500/1534
  graded 750/1534
  graded 1000/1534
  graded 1250/1534
  graded 1500/1534

sanity-gold-test_bird: 99.8% correct of 1534 (95% CI 99.54–100.0%)
  correct                   1531
  gold SQL failed              3
Saved /content/drive/MyDrive/text2sql/results/sanity-gold-test_bird.json
  graded 250/1034
  graded 500/1034
  graded 750/1034
  graded 1000/1034

sanity-gold-test_spider: 100.0% correct of 1034 (95% CI 100.0–100.0%)
  correct                   1034
Saved /content/drive/MyDrive/text2sql/results/sanity-gold-test_spider.json


In [ ]:
from src.evaluate import evaluate
RESULTS = "/content/drive/MyDrive/text2sql/results"

evaluate(tests["test_bird"], {r["id"]: r["gold_sql"] for r in tests["test_bird"]},
         paths, run_id="sanity-gold-test_bird", out_dir=RESULTS)

  graded 250/1534
  graded 500/1534
  graded 750/1534
  graded 1000/1534
  graded 1250/1534
  graded 1500/1534

sanity-gold-test_bird: 99.87% correct of 1534 (95% CI 99.67–100.0%)
  correct                   1532
  gold SQL failed              2
Saved /content/drive/MyDrive/text2sql/results/sanity-gold-test_bird.json


{'run_id': 'sanity-gold-test_bird',
 'questions': 1534,
 'accuracy': 99.87,
 'ci95': (99.67, 100.0),
 'outcomes': {'correct': 1532, 'gold SQL failed': 2},
 'per_question': [{'id': 'bird_dev-dev-0',
   'db_id': 'california_schools',
   'question': 'What is the highest eligible free rate for K-12 students in the schools in Alameda County?',
   'gold_sql': "SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1",
   'difficulty': 'simple',
   'pred_sql': "SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1",
   'correct': True,
   'outcome': 'correct',
   'error': ''},
  {'id': 'bird_dev-dev-1',
   'db_id': 'california_schools',
   'question': 'Please list the lowest three eligible free rates for students aged 5-17 in continuation schools

In [ ]:
rep = json.load(open(f"{RESULTS}/sanity-gold-test_bird.json"))
for q in rep["per_question"]:
    if q["outcome"] == "gold SQL failed":
        print(q["id"], "|", q["db_id"])
        print("  question:", q["question"])
        print("  SQL:     ", q["gold_sql"][:300])
        print("  error:   ", q["error"])
        print()

bird_dev-dev-518 | card_games
  question: Which of the play format has the highest number of banned status? Indicate the play format and the names of all the card meet the condition.
  SQL:      WITH MaxBanned AS (SELECT format, COUNT(*) AS count_banned FROM legalities WHERE status = 'Banned' GROUP BY format ORDER BY COUNT(*) DESC LIMIT 1) SELECT T2.format, T1.name FROM cards AS T1 INNER JOIN legalities AS T2 ON T2.uuid = T1.uuid INNER JOIN MaxBanned MB ON MB.format = T2.format WHERE T2.sta
  error:    timeout: interrupted

bird_dev-dev-596 | codebase_community
  question: Which users have posted the most comments. List out the user's badge?
  SQL:      SELECT Name FROM badges AS T1 INNER JOIN comments AS T2 ON T1.UserId = t2.UserId GROUP BY T2.UserId ORDER BY COUNT(T2.UserId) DESC LIMIT 1
  error:    timeout: interrupted

bird_dev-dev-701 | codebase_community
  question: Among all the posts posted by the most influential user, identify the percentage with a score above 50.
  SQL:     

Timing check (Colab CPU): dev-518 took 115 s, dev-701 took over 600 s.
Decision: keep the 30 s limit for all queries. These 2 count as wrong for every model
(best possible BIRD score: 1,532/1,534 = 99.87%).

## 1.5 Baseline: base 1.5B model, zero-shot

In [1]:
!pip install -q vllm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.5.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 whic

In [ ]:
from src.generate import load_model, generate_answers

PREDS = "/content/drive/MyDrive/text2sql/results/preds"   # answers saved on Drive
MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

llm = load_model(MODEL)
for name, recs in tests.items():
    generate_answers(llm, recs, paths, out_file=f"{PREDS}/base-1.5b-zeroshot-{name}.json")

W1126 14:35:49.281000 2755 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1126 14:35:49.283000 2755 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1126 14:35:49.286000 2755 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=2755) INFO 11-26 14:36:11 [startup.py:96] non-default args: {'dtype': 'half', 'seed': 42, 'max_model_len': 8192, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-1.5B-Instruct'}


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

(MainProcess pid=2755) INFO 11-26 14:36:35 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=2755) WARNING 11-26 14:36:35 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=2755) INFO 11-26 14:36:35 [model.py:2214] Using max model len 8192
(MainProcess pid=2755) INFO 11-26 14:36:35 [scheduler.py:354] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=2755) INFO 11-26 14:36:35 [kernel.py:517] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

... (10 outputs left out)

Answered 1534 questions in 5.4 min
Saved answers to /content/drive/MyDrive/text2sql/results/preds/base-1.5b-zeroshot-test_bird.json


Rendering prompts:   0%|          | 0/1034 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1034/1034 [01:04<00:00, 16.04it/s, est. speed input: 14416.33 toks/s, output: 529.32 toks/s]

Answered 1034 questions in 1.1 min
Saved answers to /content/drive/MyDrive/text2sql/results/preds/base-1.5b-zeroshot-test_spider.json


In [ ]:
import json, os
from src.evaluate import evaluate

RESULTS = "/content/drive/MyDrive/text2sql/results"
PREDS = f"{RESULTS}/preds"

print(os.listdir(PREDS))   # should list the two answer files from the GPU run

for name, recs in tests.items():
    preds = json.load(open(f"{PREDS}/base-1.5b-zeroshot-{name}.json"))
    evaluate(recs, preds, paths, run_id=f"base-1.5b-zeroshot-{name}", out_dir=RESULTS)

['base-1.5b-zeroshot-test_bird.json', 'base-1.5b-zeroshot-test_spider.json']
  graded 250/1534
  graded 500/1534
  graded 750/1534
  graded 1000/1534
  graded 1250/1534
  graded 1500/1534

base-1.5b-zeroshot-test_bird: 27.12% correct of 1534 (95% CI 24.9–29.27%)
  no such column/table       597
  runs but wrong result      449
  correct                    416
  other error                 42
  syntax error                27
  gold SQL failed              2
  timeout                      1
Saved /content/drive/MyDrive/text2sql/results/base-1.5b-zeroshot-test_bird.json
  graded 250/1034
  graded 500/1034
  graded 750/1034
  graded 1000/1034

base-1.5b-zeroshot-test_spider: 66.05% correct of 1034 (95% CI 63.15–68.76%)
  correct                    683
  runs but wrong result      200
  no such column/table       147
  other error                  3
  syntax error                 1
Saved /content/drive/MyDrive/text2sql/results/base-1.5b-zeroshot-test_spider.json


In [ ]:
from collections import defaultdict, Counter

rep = json.load(open(f"{RESULTS}/base-1.5b-zeroshot-test_bird.json"))
by_outcome = defaultdict(list)
for q in rep["per_question"]:
    by_outcome[q["outcome"]].append(q)

for outcome in ["no such column/table", "runs but wrong result", "other error"]:
    print(f"\n========== {outcome} ==========")
    for q in by_outcome[outcome][:3]:
        print("QUESTION:", q["question"])
        print("GOLD:    ", q["gold_sql"][:250])
        print("MODEL:   ", q["pred_sql"][:250])
        if q["error"]:
            print("ERROR:   ", q["error"])
        print()

print("Most common 'other error' messages:")
for msg, k in Counter(q["error"][:60] for q in by_outcome["other error"]).most_common(5):
    print(f"  {k:>3}  {msg}")


========== no such column/table ==========
QUESTION: What is the highest eligible free rate for K-12 students in the schools in Alameda County?
GOLD:     SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1
MODEL:    SELECT MAX(`Percent (%) Eligible Free (K-12)`) FROM frpm WHERE County = 'Alameda'
ERROR:    no such column: County

QUESTION: Please list the phone numbers of the direct charter-funded schools that are opened after 2000/1/1.
GOLD:     SELECT T2.Phone FROM frpm AS T1 INNER JOIN schools AS T2 ON T1.CDSCode = T2.CDSCode WHERE T1.`Charter Funding Type` = 'Directly funded' AND T1.`Charter School (Y/N)` = 1 AND T2.OpenDate > '2000-01-01'
MODEL:    SELECT T1.Phone FROM frpm AS T1 INNER JOIN schools AS T2 ON T1.CDSCode = T2.CDSCode WHERE T1.CharterSchool = 1 AND T2.OpenDate > '2000-01-01'
ERROR:    no such column: T1.Phone

QUESTION: How many schools wit

## 1.5b Few-shot baseline

In [ ]:
import os
from google.colab import userdata
from src.prompts import pick_fewshot_examples, save_fewshot_examples

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")      # BIRD practice data is in your private HF repo
paths_all = unpack_all(include_bird_train=True)         # adds BIRD practice databases
train = read_jsonl(DRIVE_PROCESSED / "train.jsonl")

examples = pick_fewshot_examples(train, paths_all)
for e in examples:
    print(e["why"], "|", e["source"], "/", e["db_id"])
    print("  Q:   ", e["question"])
    print("  Hint:", e["evidence"] or "(none)")
    print("  SQL: ", e["gold_sql"], "\n")

save_fewshot_examples(examples, FEWSHOT)

bird_train.zip: reconstructing file:   0%|          |  0.00B / 8.92GB            

bird_train.zip: downloading bytes:           |  0.00B            

Unpacking bird_train (big datasets take several minutes)...
✓ Paths saved to /content/data/raw/paths.json
BIRD question with a hint and a JOIN | bird_train / citeseer
  Q:    Calculate the percentage of words used in Agents class label.
  Hint: percentage = (divide(count(word_cited_id where class_label = 'Agents')), (count(word_cited_id)))*100;
  SQL:  SELECT CAST(COUNT(DISTINCT CASE WHEN T1.class_label = 'Agents' THEN T2.word_cited_id ELSE NULL END) AS REAL) * 100 / COUNT(DISTINCT T2.word_cited_id) FROM paper AS T1 INNER JOIN content AS T2 ON T1.paper_id = T2.paper_id 

Spider question with GROUP BY | spider / student_1
  Q:    For each grade, report the grade, the number of classrooms in which it is taught and the total number of students in the grade.
  Hint: (none)
  SQL:  SELECT grade ,  count(DISTINCT classroom) ,  count(*) FROM list GROUP BY grade 

Spider question with a JOIN | spider / network_2
  Q:    What are the names of all females who are friends with Zach?
  Hint: (none

In [ ]:
from src.generate import load_model, generate_answers
from src.prompts import load_fewshot_examples

examples = load_fewshot_examples(FEWSHOT)
llm = load_model(MODEL, max_model_len=12288)   # longer than 8,192: few-shot prompts carry 3 extra sheets
for name, recs in tests.items():
    generate_answers(llm, recs, paths, examples=examples,
                     out_file=f"{PREDS}/base-1.5b-fewshot-{name}.json")

W1126 23:43:21.812000 3250 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1126 23:43:21.814000 3250 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1126 23:43:21.816000 3250 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=3250) INFO 11-26 23:43:45 [startup.py:96] non-default args: {'dtype': 'half', 'seed': 42, 'max_model_len': 12288, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-1.5B-Instruct'}


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

(MainProcess pid=3250) INFO 11-26 23:44:05 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=3250) WARNING 11-26 23:44:05 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=3250) INFO 11-26 23:44:05 [model.py:2214] Using max model len 12288
(MainProcess pid=3250) INFO 11-26 23:44:05 [scheduler.py:354] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=3250) INFO 11-26 23:44:06 [kernel.py:517] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

... (10 outputs left out)

Answered 1534 questions in 7.4 min
Saved answers to /content/drive/MyDrive/text2sql/results/preds/base-1.5b-fewshot-test_bird.json
Longest prompt: 3123 tokens (+256 for the answer; limit 12288)


Rendering prompts:   0%|          | 0/1034 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1034/1034 [01:56<00:00,  8.90it/s, est. speed input: 17052.94 toks/s, output: 297.97 toks/s]

Answered 1034 questions in 2.0 min
Saved answers to /content/drive/MyDrive/text2sql/results/preds/base-1.5b-fewshot-test_spider.json


In [ ]:
from src.evaluate import evaluate

for name, recs in tests.items():
    preds = json.load(open(f"{PREDS}/base-1.5b-fewshot-{name}.json"))
    evaluate(recs, preds, paths, run_id=f"base-1.5b-fewshot-{name}", out_dir=RESULTS)

  graded 250/1534
  graded 500/1534
  graded 750/1534
  graded 1000/1534
  graded 1250/1534
  graded 1500/1534

base-1.5b-fewshot-test_bird: 22.88% correct of 1534 (95% CI 20.6–24.97%)
  no such column/table       707
  runs but wrong result      407
  correct                    351
  other error                 36
  syntax error                29
  timeout                      2
  gold SQL failed              2
Saved /content/drive/MyDrive/text2sql/results/base-1.5b-fewshot-test_bird.json
  graded 250/1034
  graded 500/1034
  graded 750/1034
  graded 1000/1034

base-1.5b-fewshot-test_spider: 66.54% correct of 1034 (95% CI 63.83–69.25%)
  correct                    688
  runs but wrong result      195
  no such column/table       138
  other error                 11
  syntax error                 2
Saved /content/drive/MyDrive/text2sql/results/base-1.5b-fewshot-test_spider.json


In [ ]:
import re

def load_run(run_id):
    return {q["id"]: q for q in json.load(open(f"{RESULTS}/{run_id}.json"))["per_question"]}

zero = load_run("base-1.5b-zeroshot-test_bird")
few = load_run("base-1.5b-fewshot-test_bird")

gained = [i for i in zero if few[i]["correct"] and not zero[i]["correct"]]
lost = [i for i in zero if zero[i]["correct"] and not few[i]["correct"]]
print(f"Correct in both: {sum(zero[i]['correct'] and few[i]['correct'] for i in zero)}")
print(f"Fixed by few-shot: {len(gained)}   Broken by few-shot: {len(lost)}\n")

# Names that exist only in the example databases
example_names = ["class_label", "word_cited_id", "classroom", "PersonFriend"]
pattern = re.compile(r"\b(" + "|".join(example_names) + r")\b", re.I)
for label, run in [("zero-shot", zero), ("few-shot", few)]:
    n = sum(bool(pattern.search(q["pred_sql"])) for q in run.values())
    print(f"{label}: {n} answers use a name from the example databases")

print("\nTwo questions few-shot broke:")
for i in lost[:2]:
    print("QUESTION: ", zero[i]["question"])
    print("ZERO-SHOT:", zero[i]["pred_sql"][:200])
    print("FEW-SHOT: ", few[i]["pred_sql"][:200], "|", few[i]["outcome"], "\n")

Correct in both: 261
Fixed by few-shot: 90   Broken by few-shot: 155

zero-shot: 0 answers use a name from the example databases
few-shot: 0 answers use a name from the example databases

Two questions few-shot broke:
QUESTION:  What is the phone number of the school that has the highest number of test takers with an SAT score of over 1500?
ZERO-SHOT: SELECT T1.Phone FROM schools AS T1 INNER JOIN satscores AS T2 ON T1.CDSCode = T2.cds WHERE T2.NumGE1500 > 0 ORDER BY T2.NumTstTakr DESC LIMIT 1
FEW-SHOT:  SELECT T1.phone FROM schools AS T1 INNER JOIN satscores AS T2 ON T1.cds  =  T2.cds WHERE T2.NumTstTakr  >  1500 ORDER BY T2.NumTstTakr DESC LIMIT 1 | no such column/table 

QUESTION:  What is the phone number of the school that has the highest average score in Math?
ZERO-SHOT: SELECT T1.Phone FROM schools AS T1 INNER JOIN satscores AS T2 ON T1.CDSCode = T2.cds ORDER BY T2.AvgScrMath DESC LIMIT 1
FEW-SHOT:  SELECT T1.phone FROM schools AS T1 INNER JOIN satscores AS T2 ON T1.cds  =  T2.cd

## 1.6 Comparison: 7B AWQ, zero-shot

In [3]:
from src.generate import load_model, generate_answers

MODEL_7B = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"
llm = load_model(MODEL_7B)              # vLLM sees the AWQ format from the model's files
for name, recs in tests.items():
    generate_answers(llm, recs, paths, out_file=f"{PREDS}/base-7b-awq-zeroshot-{name}.json")

W1127 15:11:06.613000 2193 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1127 15:11:06.614000 2193 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1127 15:11:06.616000 2193 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=2193) INFO 11-27 15:11:34 [startup.py:96] non-default args: {'dtype': 'half', 'seed': 42, 'max_model_len': 8192, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-7B-Instruct-AWQ'}


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

(MainProcess pid=2193) INFO 11-27 15:11:57 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=2193) INFO 11-27 15:11:57 [model.py:2214] Using max model len 8192
(MainProcess pid=2193) INFO 11-27 15:11:59 [scheduler.py:354] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json:   0%|          | 0.00/62.7k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

... (11 outputs left out)

Answered 1534 questions in 13.0 min
Saved answers to /content/drive/MyDrive/text2sql/results/preds/base-7b-awq-zeroshot-test_bird.json
Longest prompt: 2105 tokens (+256 for the answer; limit 8192)


Rendering prompts:   0%|          | 0/1034 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1034/1034 [02:55<00:00,  5.88it/s, est. speed input: 5288.10 toks/s, output: 191.84 toks/s]

Answered 1034 questions in 3.0 min
Saved answers to /content/drive/MyDrive/text2sql/results/preds/base-7b-awq-zeroshot-test_spider.json


In [3]:
from src.evaluate import evaluate

for name, recs in tests.items():
    preds = json.load(open(f"{PREDS}/base-7b-awq-zeroshot-{name}.json"))
    evaluate(recs, preds, paths, run_id=f"base-7b-awq-zeroshot-{name}", out_dir=RESULTS)

  graded 250/1534
  graded 500/1534
  graded 750/1534
  graded 1000/1534
  graded 1250/1534
  graded 1500/1534

base-7b-awq-zeroshot-test_bird: 48.83% correct of 1534 (95% CI 46.15–51.3%)
  correct                    749
  runs but wrong result      618
  no such column/table       132
  other error                 25
  syntax error                 7
  gold SQL failed              2
  timeout                      1
Saved /content/drive/MyDrive/text2sql/results/base-7b-awq-zeroshot-test_bird.json
  graded 250/1034
  graded 500/1034
  graded 750/1034
  graded 1000/1034

base-7b-awq-zeroshot-test_spider: 77.76% correct of 1034 (95% CI 75.15–80.37%)
  correct                    804
  runs but wrong result      199
  no such column/table        26
  other error                  5
Saved /content/drive/MyDrive/text2sql/results/base-7b-awq-zeroshot-test_spider.json
